In [1]:
# import the necessary libraries and functions
import os

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff

import hvplot.pandas
import panel as pn

from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.models import LinearColorMapper, ColorBar, BasicTicker, PrintfTickFormatter, ColumnDataSource, FactorRange
from bokeh.transform import transform

In [ ]:
# from bokeh.io import dodge

In [ ]:
# define the filepath
file_path = os.path.join('data', 'results', 'llm_predictions_results.csv')
print(file_path)

# load the dataframe
df = pd.read_csv(file_path)
df.head()

In [ ]:
# Assuming your DataFrame is named df
df = df.drop(df.columns[[0]], axis=1)
df.head()

In [ ]:
import os
print(os.getcwd())

In [ ]:
df.tail()

# Interactive Heatmaps Plotly Heatmap

*Purpose*: Visualize the interaction between dot_value and distance_counts with model_overall_accuracy.

In [ ]:
# Assuming 'pivot_table' is created using pivot_table as previously discussed

# Example pivot_table creation:
pivot_table = df.pivot_table(
    index='dot_value',
    columns='distance_counts',
    values='model_overall_accuracy',
    aggfunc='mean',
    fill_value=0
).reset_index()

# Melt the pivot table for Plotly
melted = pivot_table.melt(id_vars='dot_value', var_name='distance_counts', value_name='model_overall_accuracy')

# Create the heatmap
fig = px.density_heatmap(
    melted, 
    x='distance_counts', 
    y='dot_value', 
    z='model_overall_accuracy',
    color_continuous_scale='YlGnBu',
    title='Model Overall Accuracy Heatmap',
    labels={'distance_counts': 'Distance Counts', 'dot_value': 'Dot Value', 'model_overall_accuracy': 'Accuracy'}
)

fig.update_layout(
    xaxis_nticks=20
)

fig.show()

# Interactive Scatter Plots

Plotly Scatter Plot with Hover Information

*Purpose*: Visualize the relationship between dot_counts and distance_counts, colored by model_overall_accuracy.

In [ ]:
output_notebook()

# Prepare data
pivot_table = df.pivot_table(
    index='dot_value',
    columns='distance_counts',
    values='model_overall_accuracy',
    aggfunc='mean',
    fill_value=0
).reset_index()

melted = pivot_table.melt(id_vars='dot_value', var_name='distance_counts', value_name='model_overall_accuracy')

# Create a color mapper
mapper = LinearColorMapper(palette="Viridis256", low=melted['model_overall_accuracy'].min(), high=melted['model_overall_accuracy'].max())

# Create figure
p = figure(title="Model Overall Accuracy Heatmap", x_range=[str(x) for x in sorted(melted['distance_counts'].unique())],
           y_range=[str(y) for y in sorted(melted['dot_value'].unique())],
           x_axis_location="above", plot_width=600, plot_height=400,
           tools="hover,save,pan,box_zoom,reset,wheel_zoom",
           toolbar_location='below', tooltips=[('Distance Counts', '@x'), ('Dot Value', '@y'), ('Accuracy', '@z')])

p.grid.grid_line_color = None
p.axis.axis_line_color = None
p.axis.major_tick_line_color = None
p.axis.major_label_text_font_size = "10pt"
p.axis.major_label_standoff = 0
p.xaxis.major_label_orientation = 1.0

# Add rectangles
p.rect(x="distance_counts", y="dot_value", width=1, height=1, source=melted,
       fill_color=transform('model_overall_accuracy', mapper),
       line_color=None)

# Add color bar
color_bar = ColorBar(color_mapper=mapper, major_label_text_font_size="10px",
                     ticker=BasicTicker(desired_num_ticks=10),
                     formatter=PrintfTickFormatter(format="%0.2f"),
                     label_standoff=6, border_line_color=None, location=(0,0))
p.add_layout(color_bar, 'right')

show(p)


# Interactive Bar Charts Plotly Grouped Bar Chart

*Purpose*: Compare Tp, Fp, and Fn across different 'dot_value' or 'distance_counts'

In [ ]:

fig = px.scatter(
    df, 
    x='dot_counts', 
    y='distance_counts', 
    color='model_overall_accuracy',
    size='model_overall_accuracy',
    hover_data=['Tp', 'Fp', 'Fn'],
    color_continuous_scale='Viridis',
    title='Accuracy by Dot Counts and Distance Counts',
    labels={'dot_counts': 'Dot Counts', 'distance_counts': 'Distance Counts', 'model_overall_accuracy': 'Accuracy'}
)

fig.update_layout(coloraxis_colorbar=dict(title="Accuracy"))

fig.show()


In [ ]:
output_notebook()

source = ColumnDataSource(df)

p = figure(title="Accuracy by Dot Counts and Distance Counts",
           x_axis_label='Dot Counts', y_axis_label='Distance Counts',
           tools="pan,wheel_zoom,box_zoom,reset,hover,save",
           tooltips=[
               ("Dot Counts", "@dot_counts"),
               ("Distance Counts", "@distance_counts"),
               ("Accuracy", "@model_overall_accuracy"),
               ("Tp", "@Tp"),
               ("Fp", "@Fp"),
               ("Fn", "@Fn"),
           ])

scatter = p.scatter('dot_counts', 'distance_counts', source=source,
                   size='model_overall_accuracy', color='model_overall_accuracy',
                   fill_alpha=0.6, line_color="white")

color_mapper = LinearColorMapper(palette="Viridis256", low=df['model_overall_accuracy'].min(), high=df['model_overall_accuracy'].max())
p.scatter('dot_counts', 'distance_counts', source=source,
          size='model_overall_accuracy', color={'field': 'model_overall_accuracy', 'transform': color_mapper},
          fill_alpha=0.6, line_color="white")

color_bar = ColorBar(color_mapper=color_mapper, location=(0,0),
                     title="Accuracy")
p.add_layout(color_bar, 'right')

show(p)


In [ ]:
# Melt the DataFrame for Plotly
metrics = df.melt(id_vars=['dot_value', 'distance_counts'], value_vars=['Tp', 'Fp', 'Fn'],
                 var_name='Metric', value_name='Count')

# Create a grouped bar chart
fig = go.Figure()

metrics_unique = metrics['Metric'].unique()

for metric in metrics_unique:
    fig.add_trace(go.Bar(
        name=metric,
        x=metrics[metrics['Metric'] == metric]['dot_value'],
        y=metrics[metrics['Metric'] == metric]['Count']
    ))

fig.update_layout(
    barmode='group',
    title='Tp, Fp, Fn by Dot Value',
    xaxis_title='Dot Value',
    yaxis_title='Count',
    legend_title='Metric'
)

fig.show()


# Interactive ROC Curves

a. Plotly ROC Curve

*Purpose*: Visualize the trade-off between False Positive Rate (FPR) and True Positive Rate (TPR).

In [ ]:
# Filter and copy the DataFrame to avoid SettingWithCopyWarning
df_filtered = df[(df['dot_value'] == 0.9) & (df['dot_counts'] == 15)].copy()

# Calculate FPR and TPR
df_filtered['FPR'] = df_filtered['Fp'] / (df_filtered['Fp'] + df_filtered['Tp'])
df_filtered['TPR'] = df_filtered['Tp'] / (df_filtered['Tp'] + df_filtered['Fn'])

# Sort by FPR for a proper ROC curve
df_filtered = df_filtered.sort_values(by='FPR')

# Create ROC curve
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_filtered['FPR'],
    y=df_filtered['TPR'],
    mode='lines+markers',
    name='ROC Curve',
    hovertemplate='FPR: %{x:.2f}<br>TPR: %{y:.2f}<extra></extra>'
))

fig.add_trace(go.Scatter(
    x=[0, 1],
    y=[0, 1],
    mode='lines',
    name='Random Guess',
    line=dict(dash='dash')
))

fig.update_layout(
    title='ROC Curve for Dot Value = 0.9 and Dot Counts = 15',
    xaxis_title='False Positive Rate (FPR)',
    yaxis_title='True Positive Rate (TPR)',
    xaxis=dict(range=[0,1]),
    yaxis=dict(range=[0,1]),
    legend=dict(x=0.6, y=0.1)
)

fig.show()


# Interactive 3D Surface Plots 

a. Plotly 3D Surface Plot

*Purpose*: Visualize how model_overall_accuracy varies over dot_counts and distance_counts.

In [ ]:
# Create pivot table
pivot_table = df.pivot_table(
    index='dot_counts',
    columns='distance_counts',
    values='model_overall_accuracy',
    aggfunc='mean',
    fill_value=0
)

X, Y = np.meshgrid(pivot_table.columns, pivot_table.index)
Z = pivot_table.values

fig = go.Figure(data=[go.Surface(z=Z, x=X, y=Y, colorscale='Viridis')])

fig.update_layout(
    title='Model Overall Accuracy Surface Plot',
    scene=dict(
        xaxis_title='Distance Counts',
        yaxis_title='Dot Counts',
        zaxis_title='Accuracy'
    )
)

fig.show()

In [ ]:
pn.extension('plotly')

# Assuming 'pivot_table' is created as above
pivot_table = df.pivot_table(
    index='dot_counts',
    columns='distance_counts',
    values='model_overall_accuracy',
    aggfunc='mean',
    fill_value=0
)

# Create a 3D scatter plot (as Bokeh doesn't support surface plots directly)
hv_scatter = pivot_table.reset_index().melt(id_vars='dot_counts', var_name='distance_counts', value_name='model_overall_accuracy').hvplot.scatter(
    x='dot_counts', 
    y='distance_counts', 
    z='model_overall_accuracy', 
    size='model_overall_accuracy',
    color='model_overall_accuracy',
    cmap='Viridis',
    title='Model Overall Accuracy Scatter Plot',
    xlabel='Dot Counts',
    ylabel='Distance Counts',
    tools=['hover', 'box_zoom', 'reset']
)

pn.panel(hv_scatter).servable()

# Interactive Correlation Matrix

a. Plotly Correlation Heatmap with Interactive Features

*Purpose*: Explore the correlations between different variables interactively.

In [ ]:
# Compute correlation matrix
corr = df.corr()

# Melt the correlation matrix for Plotly
corr_melted = corr.reset_index().melt(id_vars='index', var_name='Variable_2', value_name='Correlation')
corr_melted.rename(columns={'index': 'Variable_1'}, inplace=True)

# Create the heatmap
fig = px.imshow(
    corr,
    text_auto=True,
    color_continuous_scale='RdBu',
    zmin=-1, zmax=1,
    title='Correlation Matrix of Key Variables',
    labels=dict(x="Variables", y="Variables", color="Correlation")
)

fig.update_layout(
    xaxis_title='Variables',
    yaxis_title='Variables'
)

fig.show()


# Interactive Radar (Spider) Charts

While neither Plotly nor Bokeh natively support radar charts as straightforwardly as other plot types, Plotly offers a way to create them using scatterpolar.

a. Plotly Radar Chart

*Purpose*: Compare multiple performance metrics across different parameter settings.

In [ ]:
# Select a specific dot_value or distance_counts to compare metrics
# For illustration, we'll compare metrics at different distance_counts for a fixed dot_value

dot_val = 0.9
metrics_subset = df[df['dot_value'] == dot_val].groupby('distance_counts').mean().reset_index()

categories = ['model_overall_accuracy', 'precision', 'recall', 'llm_activation_sensitivity', 'llm_interaction_accuracy']

fig = go.Figure()

for _, row in metrics_subset.iterrows():
    values = row[categories].tolist()
    values += values[:1]  # Complete the loop
    fig.add_trace(go.Scatterpolar(
        r=values,
        theta=categories + [categories[0]],
        fill='toself',
        name=f'Distance Counts: {row["distance_counts"]}'
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 1]
        )
    ),
    title=f'Performance Metrics Radar Chart for Dot Value = {dot_val}',
    showlegend=True
)

fig.show()



# Interactive Scatter Matrix (Pair Plot a. Plotly Scatter Matrix

*Purpose*: Explore pairwise relationships and correlations between multiple variables interactively

In [ ]:
# Select relevant columns
corr_cols = [
    'dot_counts', 'distance_counts',
    'model_overall_accuracy', 'precision', 'recall',
    'llm_activation_sensitivity', 'llm_interaction_accuracy',
    'Tp', 'Fp', 'Fn'
]

fig = px.scatter_matrix(
    df,
    dimensions=corr_cols,
    color='model_overall_accuracy',
    title='Scatter Matrix of Key Variables',
    labels={col: col.replace('_', ' ').title() for col in corr_cols},
    color_continuous_scale='Viridis',
    opacity=0.7
)

fig.update_traces(diagonal_visible=False)
fig.show()


# Interactive Time Series Plots

If your time variable represents different time points or sequences, interactive time series plots can help visualize performance metrics over time.

In [ ]:
# Assume 'time' is a sequential variable indicating different time points
fig = go.Figure()

metrics = ['model_overall_accuracy', 'precision', 'recall', 'llm_activation_sensitivity', 'llm_interaction_accuracy']

for metric in metrics:
    fig.add_trace(go.Scatter(
        x=df['time'],
        y=df[metric],
        mode='lines+markers',
        name=metric.replace('_', ' ').title(),
        hovertemplate='Time: %{x}<br>' + metric.replace('_', ' ').title() + ': %{y:.2f}<extra></extra>'
    ))

fig.update_layout(
    title='Performance Metrics Over Time',
    xaxis_title='Time',
    yaxis_title='Metric Values',
    hovermode='closest'
)

fig.show()

# Interactive Confusion Matrix

a. Plotly Confusion Matrix Heatmap with Annotations

*Purpose*: Visualize the confusion matrix components (Tp, Fp, Fn, Tn) interactively.

In [ ]:
# Select a specific parameter setting
cm_row = df_filtered.iloc[0]

# Assuming 'Tn' (True Negatives) is calculated as follows:
cm_row['Tn'] = cm_row['Total_ground_truths'] - cm_row['Tp'] - cm_row['Fn']

# Create confusion matrix data
z = [
    [cm_row['Tp'], cm_row['Fp']],
    [cm_row['Fn'], cm_row['Tn']]
]
x = ['Predicted Positive', 'Predicted Negative']
y = ['Actual Positive', 'Actual Negative']

fig = ff.create_annotated_heatmap(z, x=x, y=y, colorscale='Blues', showscale=True)

fig.update_layout(
    title='Confusion Matrix',
    xaxis_title='Predicted Label',
    yaxis_title='Actual Label'
)

fig.show()
